# 01 — From Text to Next-Token Prediction

**Description:** Build a tiny language model from text, inspect its possible next tokens, and generate new text one token at a time.
**Level:** Beginner
**Tags:** Language Models, GPT, Next-Token Prediction, Generation

A language model answers one repeated question: **given the tokens so far, what token might come next?** In this notebook, you will build a small model whose behavior is completely visible. It is not a neural network, but it uses the same predict-and-sample loop as GPT-style models.

By the end, you will be able to:

- turn text into tokens;
- learn next-token probabilities from examples;
- inspect the distribution for a short context; and
- generate text by repeatedly sampling one next token.

In [ ]:
import re
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(7)  # Reproducible sampling
plt.style.use("seaborn-v0_8-whitegrid")

## 1. Start with examples

Models learn patterns from data. Our tiny training corpus deliberately repeats sentence structures so that useful patterns appear even with very little text. Each line is one training example.

Read a few lines and predict what might follow `the robot`. Your intuition is already assigning different probabilities to different continuations.

In [ ]:
corpus = [
    "the robot learns from examples",
    "the robot learns from data",
    "the robot learns quickly",
    "the robot writes a poem",
    "the robot writes a story",
    "the student learns from examples",
    "the student learns from mistakes",
    "the student writes a story",
    "a model learns from data",
    "a model predicts the next token",
    "a model writes one token at a time",
    "a language model predicts a token",
]

print(f"Training examples: {len(corpus)}")
for sentence in corpus[:5]:
    print(" •", sentence)

## 2. Text becomes tokens

A model does not operate directly on a Python string. It receives **tokens**: discrete pieces of text. Real GPT tokenizers may split rare words into smaller pieces. To keep the mechanics visible, our tokenizer uses lowercase words and punctuation.

We also add two special tokens: `<BOS>` marks the beginning of a sequence and `<EOS>` marks its end.

In [ ]:
def tokenize(text):
    """Split text into lowercase word and punctuation tokens."""
    return re.findall(r"[a-z]+|[^\w\s]", text.lower())

example = "The robot writes a poem!"
tokens = tokenize(example)
print("Text:  ", example)
print("Tokens:", tokens)
print("Count: ", len(tokens))

### Try it

Change `your_text` and rerun the cell. What happens to capitalization and punctuation? Which parts of this simple tokenizer would be inadequate for real-world text?

In [ ]:
your_text = "Language models predict, then sample."
tokenize(your_text)

## 3. Create the prediction task

Training examples can be made from every position in a sentence. With a context length of two, `the robot learns` becomes:

| Context | Target (the answer) |
|---|---|
| `<BOS> <BOS>` | `the` |
| `<BOS> the` | `robot` |
| `the robot` | `learns` |
| `robot learns` | `<EOS>` or the following word |

The target is shifted one position ahead of the context. Neural language models learn from the same kind of input/target alignment.

In [ ]:
def training_pairs(sentence, context_size=2):
    sequence = ["<BOS>"] * context_size + tokenize(sentence) + ["<EOS>"]
    return [
        (tuple(sequence[i - context_size:i]), sequence[i])
        for i in range(context_size, len(sequence))
    ]

pairs = training_pairs("the robot learns quickly")
for context, target in pairs:
    print(f"{str(context):27s} → {target}")

## 4. Learn by counting

Our model is an **n-gram language model**. It counts which targets occurred after each context. Counts become probabilities by dividing by their total:

$$P(\text{next token} \mid \text{context}) = \frac{\text{count(context, next token)}}{\sum_t \text{count(context, }t\text{)}}$$

For example, if `learns` appears three times and `writes` twice after `the robot`, their probabilities are $3/5$ and $2/5$.

In [ ]:
def fit_ngram_model(sentences, max_context=2):
    # Store counts for context lengths 0, 1, and 2. Shorter contexts
    # provide a fallback when the exact two-token context was never seen.
    counts = {size: defaultdict(Counter) for size in range(max_context + 1)}
    for sentence in sentences:
        sequence = ["<BOS>"] * max_context + tokenize(sentence) + ["<EOS>"]
        for i in range(max_context, len(sequence)):
            target = sequence[i]
            for size in range(max_context + 1):
                context = tuple(sequence[i - size:i]) if size else ()
                counts[size][context][target] += 1
    return counts

model = fit_ngram_model(corpus)
model[2][("the", "robot")]

## 5. Inspect a next-token distribution

The output is not a single word. It is a **probability distribution** over possible next tokens. The most likely token is a reasonable prediction, but alternatives matter: they are what make multiple continuations possible.

If the exact two-token context was never observed, `predict_next` backs off to the last one token, then to overall token frequencies. This is a simple solution to unfamiliar contexts; modern models instead learn representations that generalize.

In [ ]:
def predict_next(text, counts=model, max_context=2):
    tokens = tokenize(text)
    for size in range(min(max_context, len(tokens)), -1, -1):
        context = tuple(tokens[-size:]) if size else ()
        next_counts = counts[size].get(context)
        if next_counts:
            words = np.array(list(next_counts))
            values = np.array(list(next_counts.values()), dtype=float)
            probabilities = values / values.sum()
            order = np.argsort(probabilities)[::-1]
            return context, words[order], probabilities[order]
    raise RuntimeError("The model contains no tokens.")

context, words, probabilities = predict_next("the robot")
print("Context used:", context)
for word, probability in zip(words, probabilities):
    print(f"{word:>10s}  {probability:6.1%}")

### Make the distribution visible

A bar chart makes the model's uncertainty easier to see. A peaked distribution expresses a strong preference; a flatter one leaves several plausible choices.

In [ ]:
def plot_next_tokens(text):
    context, words, probabilities = predict_next(text)
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.bar(words, probabilities, color="#4C78A8")
    ax.set(title=f"Next-token probabilities after: {' '.join(context) or '<any context>'}",
           ylabel="Probability", ylim=(0, 1))
    ax.tick_params(axis="x", rotation=30)
    plt.show()

plot_next_tokens("the robot")

### Your turn: change the context

Try `the student`, `learns from`, and a context absent from the corpus such as `my robot`. Before running the cell, write down your expected top token. Also inspect `context_used` to see whether the model used two tokens, one token, or the fallback distribution.

In [ ]:
my_context = "learns from"  # Edit me
context_used, candidate_tokens, candidate_probabilities = predict_next(my_context)
print("Context used:", context_used)
list(zip(candidate_tokens.tolist(), candidate_probabilities.round(3).tolist()))

## 6. Prediction is not generation

A probability distribution becomes text only after we choose a token. Two common strategies are:

- **Greedy decoding:** always choose the highest-probability token. It is deterministic.
- **Sampling:** randomly choose according to the probabilities. Likely tokens are selected more often, but lower-probability tokens sometimes appear.

Run the next cell several times after changing the seed or removing the seed reset.

In [ ]:
context, words, probabilities = predict_next("the robot")
greedy_choice = words[0]
sampled_choices = rng.choice(words, size=12, p=probabilities)

print("Greedy choice:  ", greedy_choice)
print("Sampled choices:", ", ".join(sampled_choices))

## 7. Generate one token at a time

Generation is a loop:

1. predict probabilities for the next token;
2. select one token;
3. append it to the context;
4. repeat using the longer context.

This is called **autoregressive generation** because each new prediction depends on tokens generated earlier. GPT stands for *Generative Pre-trained Transformer*. A GPT uses a transformer neural network to compute much richer probabilities, but generation still follows this loop.

In [ ]:
def generate(prompt, max_new_tokens=12, strategy="sample", show_steps=False):
    generated = tokenize(prompt)
    for step in range(max_new_tokens):
        text_so_far = " ".join(generated)
        context, words, probabilities = predict_next(text_so_far)
        if strategy == "greedy":
            next_token = words[0]
        elif strategy == "sample":
            next_token = rng.choice(words, p=probabilities)
        else:
            raise ValueError("strategy must be 'greedy' or 'sample'")
        if show_steps:
            print(f"step {step + 1:2d} | {context!s:25s} → {next_token}")
        if next_token == "<EOS>":
            break
        generated.append(str(next_token))
    return " ".join(generated)

generate("the robot", strategy="sample", show_steps=True)

### Compare decoding strategies

Greedy decoding repeats the same result for a given prompt. Sampling can follow different branches. Generate several continuations and compare them. With such a small corpus, many generations will reproduce fragments of training sentences—that limitation is useful to notice.

In [ ]:
prompt = "a model"  # Try: "the robot", "the student", or "a language"

print("Greedy:")
print(" ", generate(prompt, strategy="greedy"))
print("\nSamples:")
for _ in range(5):
    print(" ", generate(prompt, strategy="sample"))

## 8. Experiment: change what the model learns

The learned distribution reflects the training data. Add several new sentences beginning with `the robot` and refit the model. Then check how the probabilities change. This is a tiny demonstration of a major principle: **a model's behavior depends on the examples and patterns in its data.**

In [ ]:
expanded_corpus = corpus + [
    "the robot dances slowly",
    "the robot dances quickly",
    # Add more examples here. Repetition changes the learned frequency.
]

expanded_model = fit_ngram_model(expanded_corpus)
_, words, probabilities = predict_next("the robot", counts=expanded_model)
list(zip(words.tolist(), probabilities.round(3).tolist()))

## 9. What this toy model captures—and what it misses

Our model and GPT-style models share the central task: estimate a next-token distribution, choose a token, append it, and repeat. But the way they estimate probabilities is very different.

| This notebook's model | GPT-style language model |
|---|---|
| Counts exact short contexts | Learns neural-network weights |
| Uses at most two previous tokens | Can use a much longer context |
| Has a tiny word-level vocabulary | Usually uses subword tokens |
| Cannot understand similar unseen contexts | Learns representations that can generalize |
| Trains on 12 sentences | Trains on very large text collections |

The next notebooks will open up the pieces hidden inside the neural version: tensors, weights, embeddings, logits, softmax, and sampling controls.

## 10. Challenges

Use the existing functions to investigate these questions:

1. **Data:** Add examples so that `the robot` is followed by `explores` more than 50% of the time. Verify the probability.
2. **Context:** Compare predictions after `model` and `a model`. Why can they differ?
3. **Generation:** Generate 20 samples from the same prompt. Count how often each first new token appears. Do the observed frequencies resemble the predicted probabilities?
4. **Reasoning:** Find a prompt for which the model backs off to a one-token context. Find one for which it uses the zero-token fallback.
5. **Extension:** Modify `generate` so it returns the probability of every selected token along with the text.

In [ ]:
# Challenge workspace
challenge_prompt = "the robot"
context_used, words, probabilities = predict_next(challenge_prompt)
print("Context used:", context_used)
print("Distribution:", dict(zip(words, probabilities.round(3))))

## Takeaways

- A language model maps a context to probabilities for the next token.
- Training creates input/target pairs by shifting text one token forward.
- Generation is repeated next-token prediction, not a whole answer produced at once.
- Greedy decoding chooses the most likely token; sampling creates varied continuations.
- The training data and available context shape the model's predictions.

**Next:** *02 — Tensors, Weights, and Layers* will replace counts with numerical operations that neural networks can learn.